<a href="https://colab.research.google.com/github/Vishal-Kotha/P9d21-OpenWork_Cowork/blob/main/Audio_to_md.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Audio → Markdown Transcript (faster-whisper + optional speaker diarization)

Transcribes audio/video recordings (interviews, lab meetings, lecture recordings) into a clean, \
speaker-labeled Markdown transcript.

**Rebuilt from your original notebook — two things were fixed:**
1. The old notebook had `!pip install git+https://github.com` with **no repository path** — that \
install was broken and would have failed immediately.
2. A **real Hugging Face access token was hardcoded in plaintext** in the old notebook. That is a \
live credential leak — if you haven't already, revoke it at \
[huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) and generate a new one. \
This notebook reads the token from Colab Secrets instead, so it never appears in the file.

**Engine choice:** [faster-whisper](https://github.com/SYSTRAN/faster-whisper) (CTranslate2 \
reimplementation of Whisper) instead of plain `openai-whisper` — same accuracy, several times faster \
on Colab's free T4 GPU, and lower memory use.


### Step 1 — Configure

- `ENABLE_DIARIZATION`: label *who* said each line, not just *what* was said. Requires a Hugging Face \
token with access granted to two gated models — see the note below.
- `MODEL_SIZE`: `"large-v3"` for best accuracy, `"distil-large-v3"` for ~2x faster with a small accuracy \
trade-off, `"medium"`/`"small"` if you're not on a GPU runtime.
- `LANGUAGE`: an ISO code like `"en"`, or `None` to auto-detect.
- `INITIAL_PROMPT`: optional domain vocabulary to bias decoding — e.g. a comma-separated list of \
technical terms, chemical names, or acronyms that come up in your recordings, so Whisper is more \
likely to spell them correctly.
- `BEAM_SIZE`: higher (e.g. 5) is more accurate but slower; 1 is fastest (greedy decoding).
- `MIN_SPEAKERS` / `MAX_SPEAKERS`: if you know how many people are in the recording, set both to that \
number (or a tight range) — diarization accuracy improves a lot with this hint. Leave `None` to let \
pyannote guess.

**If you enable diarization**, before running this notebook:
1. Add a Colab Secret named `HF_TOKEN` (key icon, left sidebar) with a Hugging Face access token, and \
grant this notebook access.
2. Visit [huggingface.co/pyannote/speaker-diarization-3.1](https://huggingface.co/pyannote/speaker-diarization-3.1) \
and [huggingface.co/pyannote/segmentation-3.0](https://huggingface.co/pyannote/segmentation-3.0) while \
logged in, and accept each model's user conditions (both are free, just gated behind a click-through).


In [ ]:
# ---- User configuration ----
ENABLE_DIARIZATION = False
MODEL_SIZE = "large-v3"     # "large-v3" | "distil-large-v3" | "medium" | "small"
LANGUAGE = None              # e.g. "en", or None to auto-detect
INITIAL_PROMPT = ""          # e.g. "Tafel slope, EIS, Nyquist, overpotential, ferrocyanide"
BEAM_SIZE = 5
MIN_SPEAKERS = None          # e.g. 2, if you know the exact/minimum speaker count
MAX_SPEAKERS = None          # e.g. 2, if you know the exact/maximum speaker count
# -----------------------------


### Step 2 — Install dependencies

In [ ]:
!apt-get -qq install -y ffmpeg
!pip install -q faster-whisper

if ENABLE_DIARIZATION:
    !pip install -q "pyannote.audio>=3.1"


### Step 3 — Load the models

In [ ]:
import torch
from faster_whisper import WhisperModel

device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "int8"
print(f"Loading Whisper ({MODEL_SIZE}) on {device} [{compute_type}]...")
whisper_model = WhisperModel(MODEL_SIZE, device=device, compute_type=compute_type)

diar_pipeline = None
if ENABLE_DIARIZATION:
    from google.colab import userdata
    from pyannote.audio import Pipeline

    hf_token = userdata.get("HF_TOKEN")
    print("Loading speaker diarization pipeline...")
    diar_pipeline = Pipeline.from_pretrained(
        "pyannote/speaker-diarization-3.1", token=hf_token
    )
    if device == "cuda":
        diar_pipeline.to(torch.device("cuda"))

print("Models ready.")


### Step 4 — Transcription function

In [ ]:
from pathlib import Path

def format_timestamp(seconds: float) -> str:
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    return f"{h:02d}:{m:02d}:{s:02d}" if h else f"{m:02d}:{s:02d}"

def assign_speaker(t_start, t_end, turns):
    """Pick the diarization speaker with the largest time overlap. If nothing overlaps
    (a gap between diarization turns), fall back to whichever turn is nearest in time
    rather than emitting an UNKNOWN label."""
    if not turns:
        return "UNKNOWN"
    best_speaker, best_overlap = None, 0.0
    for ts, te, spk in turns:
        overlap = max(0.0, min(t_end, te) - max(t_start, ts))
        if overlap > best_overlap:
            best_overlap, best_speaker = overlap, spk
    if best_speaker is not None:
        return best_speaker
    midpoint = (t_start + t_end) / 2
    nearest = min(turns, key=lambda t: min(abs(midpoint - t[0]), abs(midpoint - t[1])))
    return nearest[2]

def transcribe_one(audio_path: Path) -> str:
    print(f"Transcribing: {audio_path.name}")
    segments, info = whisper_model.transcribe(
        str(audio_path),
        language=LANGUAGE,
        vad_filter=True,                     # skip silence instead of hallucinating text over it
        beam_size=BEAM_SIZE,
        initial_prompt=INITIAL_PROMPT or None,
        word_timestamps=ENABLE_DIARIZATION,  # only needed for per-word speaker assignment
    )
    segments = list(segments)
    print(f"  detected language: {info.language} (p={info.language_probability:.2f}), "
          f"{len(segments)} segments")

    turns = []
    if diar_pipeline is not None:
        print("  running speaker diarization...")
        diar_kwargs = {}
        if MIN_SPEAKERS is not None:
            diar_kwargs["min_speakers"] = MIN_SPEAKERS
        if MAX_SPEAKERS is not None:
            diar_kwargs["max_speakers"] = MAX_SPEAKERS
        diarization = diar_pipeline(str(audio_path), **diar_kwargs)
        turns = [(turn.start, turn.end, speaker)
                 for turn, _, speaker in diarization.itertracks(yield_label=True)]

    lines = [f"# Transcript: {audio_path.stem}\n"]
    lines.append(f"*Language: {info.language} · Duration: {format_timestamp(info.duration)}*\n")

    current_speaker = None
    if turns:
        # Assign speakers per-word (not per-segment): a Whisper segment can span a
        # speaker change, so segment-level assignment would misattribute half of it.
        for seg in segments:
            words = seg.words or []
            if not words:
                # Word-level alignment can fail (DTW) even with word_timestamps=True;
                # fall back to segment-level assignment so the text isn't silently dropped.
                text = seg.text.strip()
                if not text:
                    continue
                speaker = assign_speaker(seg.start, seg.end, turns)
                if speaker != current_speaker:
                    lines.append(f"\n\n### {speaker} ({format_timestamp(seg.start)})\n")
                    current_speaker = speaker
                lines.append(" " + text)
                continue
            for w in words:
                speaker = assign_speaker(w.start, w.end, turns)
                if speaker != current_speaker:
                    lines.append(f"\n\n### {speaker} ({format_timestamp(w.start)})\n")
                    current_speaker = speaker
                lines.append(w.word)
    else:
        for seg in segments:
            text = seg.text.strip()
            if not text:
                continue
            lines.append(f"\n**[{format_timestamp(seg.start)}]** {text}\n")

    return "".join(lines).strip() + "\n"

print("Transcriber ready.")


### Step 5 — Upload audio and run

In [ ]:
import traceback

from google.colab import files

print("Select audio/video file(s) (mp3, wav, m4a, mp4, etc.):")
uploaded = files.upload()

out_dir = Path("transcripts")
out_dir.mkdir(exist_ok=True)
results = []

if not uploaded:
    print("No files selected - nothing to do.")
else:
    seen_stems = {}
    for name in uploaded.keys():
        path = Path(name)
        n = seen_stems.get(path.stem, 0)
        seen_stems[path.stem] = n + 1
        if n > 0:
            new_path = path.with_stem(f"{path.stem}_{n}")
            path.rename(new_path)
            path = new_path
            print(f"Note: renamed to {path.name} to avoid overwriting an earlier file with the same name.")

        try:
            transcript_md = transcribe_one(path)
        except Exception:
            print(f"FAILED on {name}:")
            traceback.print_exc()
            continue

        out_path = out_dir / f"{path.stem}.md"
        out_path.write_text(transcript_md, encoding="utf-8")
        results.append(out_path)
        print(f"  done -> {out_path.name}\n")

    print(f"Transcribed {len(results)} file(s) into '{out_dir}/'.")


### Step 6 — Download everything as a zip

In [ ]:
import shutil

if results:
    zip_base = "transcripts"
    shutil.make_archive(zip_base, "zip", out_dir)
    print(f"Downloading {zip_base}.zip...")
    files.download(f"{zip_base}.zip")
else:
    print("Nothing to download - no files were successfully transcribed.")
